# Risky Decisions and Happiness (The Great Brain Experiment) — Starter Notebook

Data from a smartphone game, *What makes me happy?*, in The Great Brain Experiment app, collected 2013–2015 and shared on Dryad by Robb Rutledge. 47,067 players made choices between a **sure amount** and a **50/50 gamble**, and every 2–3 trials rated **"How happy are you right now?"** (0–100). Each play has 30 trials: 11 gain, 8 mixed, and 11 loss trials. 1,858 players also completed a depression questionnaire (BDI-II).

**How this notebook is organized**

| Section | What it does |
|---|---|
| Settings | Where the data lives (the only cell you might edit) |
| Load Data | Imports libraries, loads and prepares the data |
| Inspect the Dataset | First rows, size, column types |
| Describe Variables | Summary statistics |
| Check Missing Data | What's missing and why it matters |
| Basic Visualizations | A few plots to get oriented |
| Variables You May Want to Explore | Outcomes, predictors, grouping and time variables |
| Your Analysis Starts Here | Your space for HW 1, HW 2, and HW 3 |

Cells marked 🧰 are **starter code** written by the instructor. Cells marked ✏️ are **yours**.

Open the table of contents (the ☰ icon on the left in Colab) to jump between sections.

## Settings

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

If the default link stops working, or you downloaded the file yourself, change the path below. See the dataset guide for where to get the data.

In [ ]:
# Settings: where the data lives.
# By default the data load from this course's GitHub repository, so there's nothing to download.
REPO = "cogs118d/datasets"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/data/05_rutledge_happiness/"

# Cloned the repo and working on your own computer? Use the local folder instead:
# BASE_URL = "../data/05_rutledge_happiness/"
# Saved the files to Google Drive? Point BASE_URL at that folder, e.g.:
# BASE_URL = "/content/drive/MyDrive/05_rutledge_happiness/"

# The full data have 2.7 million trials. Set a number to keep a random sample of players
# (everyone with a depression score is always kept), or None to keep everyone.
N_PLAYERS = 5000
RANDOM_SEED = 118

# Load Data

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Libraries used in every COGS 118D starter notebook
import numpy as np                        # numbers and arrays
import pandas as pd                       # data tables ("DataFrames")
import matplotlib.pyplot as plt           # plots
import statsmodels.formula.api as smf     # GLMs and mixed models (you'll use this later)

pd.set_option("display.max_columns", 60)  # show more columns when printing tables
plt.rcParams["figure.figsize"] = (7, 4)   # default plot size

In [ ]:
# Three tables, converted from the original MATLAB file (see scripts/convert_rutledge.py in the repo)
trials_all = pd.read_csv(BASE_URL + "rutledge_trials.csv.gz")   # one row per trial, all 47,067 players (~30 s to load)
people_all = pd.read_csv(BASE_URL + "rutledge_people.csv")      # one row per player
dep = pd.read_csv(BASE_URL + "rutledge_depression.csv")         # 1,858 players with a depression score (BDI-II)
print("trials:", trials_all.shape, "| players:", len(people_all), "| with depression scores:", len(dep))

## Preprocessing

We keep a random sample of players (plus everyone with a depression score), then turn the README's numeric codes into readable labels. You end up with:

* `people`: one row per player (age group, gender, education, region, device, life satisfaction, number of plays)
* `df`: one row per trial (the choice, the outcome, and the happiness rating when one was asked)
* `dep`: one row per player with a depression score

In [ ]:
# Which players to keep
if N_PLAYERS is None:
    keep_ids = set(people_all["player"])
else:
    rng = np.random.default_rng(RANDOM_SEED)
    keep_ids = set(rng.choice(people_all["player"], size=N_PLAYERS, replace=False)) | set(dep["player"])
print("Players kept:", len(keep_ids))

In [ ]:
# Codes from the dataset README (README_Rutledge_GBE_risk_data.pdf, in data/05_rutledge_happiness/)
age_labels = {1: "18-24", 2: "25-29", 3: "30-39", 4: "40-49", 5: "50-59", 6: "60-69", 7: "70+"}
education_labels = {0: "school (GCSE)", 1: "school (A-levels/vocational)", 2: "university degree", 3: "advanced degree"}

def region(code):
    """Turn the README's location codes into broad regions (0-7 UK, 100-309 Europe, 400 USA, ...)."""
    if code <= 7: return "UK"
    if code <= 309: return "Other Europe"
    if code == 400: return "USA"
    if code <= 509: return "Other Americas"
    if code <= 909: return "Africa/Asia/Pacific"
    return "unknown (not UK)"   # 999

people = people_all[people_all["player"].isin(keep_ids)].copy()
people["age_group"] = people["age_code"].map(age_labels)
people["education"] = people["education_code"].map(education_labels)
people["region"] = people["location_code"].apply(region)

df = trials_all[trials_all["player"].isin(keep_ids)].reset_index(drop=True)
del trials_all   # free memory
print("people:", people.shape, "| trials (df):", df.shape)

In [ ]:
# Useful derived columns (all defined in the published papers)
df["trial_type"] = np.select([df["sure_amount"] > 0, df["sure_amount"] < 0], ["gain", "loss"], "mixed")
df["gamble_ev"] = (df["gamble_win"] + df["gamble_lose"]) / 2          # average value of the gamble
df["ev_advantage"] = df["gamble_ev"] - df["sure_amount"]              # how much better the gamble is on average
# Reward prediction error: how much better or worse the outcome was than expected (0 if the sure amount was taken)
df["rpe"] = np.where(df["chose_gamble"] == 1, df["outcome"] - df["gamble_ev"], 0.0)

df = df.merge(people[["player", "age_group", "age_code", "female"]], on="player", how="left")
df.head()

# Inspect the Dataset

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# The first 5 rows: each row is one observation
df.head()

In [ ]:
# Dimensions: (number of rows, number of columns)
print("Rows, columns:", df.shape)

In [ ]:
# Column names and data types (numbers, text, dates, True/False)
df.info()

In [ ]:
print("Trials per play:", df.groupby(["player", "play"]).size().median())
print("Happiness ratings per play:", df.groupby(["player", "play"])["happiness"].count().median())

# Describe Variables

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
df[["sure_amount", "gamble_win", "gamble_lose", "chose_gamble", "outcome", "choice_rt", "happiness"]].describe().round(2)

In [ ]:
print(people["age_group"].value_counts().sort_index(), "\n")
print(people["device"].value_counts(), "\n")
print(people["region"].value_counts(), "\n")
print(people["life_satisfaction"].value_counts().sort_index(), "\n")
print(dep["bdi_total"].describe().round(1))

# Check Missing Data

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Count and percent of missing values in each column (only columns with any missing)
missing = df.isna().sum()
missing_table = pd.DataFrame({"n_missing": missing, "percent": (100 * missing / len(df)).round(1)})
missing_table[missing_table["n_missing"] > 0].sort_values("n_missing", ascending=False)

**What to notice.** `happiness` is missing on most trials by design: players were asked only every 2–3 trials (12 times per play). Three things aren't visible in the missing-data table:

* `region` is `unknown (not UK)` for many early players, who could only answer "rest of the world".
* `choice_rt` has a few impossible values (negative, or hours long). Filter before analyzing response times.
* `life_satisfaction` (roughly 0–10) is not described in the README's field table, so its exact wording is unknown.
* Players who updated the app had their survey answers overwritten, and the gamble amounts changed between `design_version` 1–2 and 3–5. Consider controlling for `design_version`.

# Basic Visualizations

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

These plots are for getting oriented. They are not answers to any homework question.

In [ ]:
# How often people gamble on each trial type
df.groupby("trial_type")["chose_gamble"].mean().plot(kind="bar")
plt.ylabel("Proportion choosing the gamble")
plt.title("Gambling by trial type")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Gambling as the gamble's average advantage grows (gain trials only; bins of 10 points)
gain = df[df["trial_type"] == "gain"].copy()
gain["adv_bin"] = (gain["ev_advantage"] / 10).round() * 10
by_adv = gain.groupby("adv_bin")["chose_gamble"].mean()
plt.plot(by_adv.index, by_adv.values, marker="o")
plt.xlabel("Gamble's average value minus sure amount (points)")
plt.ylabel("Proportion choosing the gamble")
plt.title("Gain trials")
plt.show()

In [ ]:
# Happiness ratings (0-100)
plt.hist(df["happiness"].dropna(), bins=50, edgecolor="white")
plt.xlabel("Happiness rating")
plt.ylabel("Number of ratings")
plt.title("Happiness")
plt.show()

In [ ]:
# Depression scores (BDI-II, 0-63) in the subset who completed it
plt.hist(dep["bdi_total"], bins=30, edgecolor="white")
plt.xlabel("BDI-II total")
plt.ylabel("Number of players")
plt.title("Depression scores")
plt.show()

# Variables You May Want to Explore

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

**Outcome variables:** `chose_gamble` (yes/no), `happiness` (0–100), `choice_rt`; per player, mean happiness or proportion of gambles.

**Predictors:** `trial_type`, `ev_advantage`, `outcome`, `rpe`, `trial`, `age_group`, `female`, `device`; `bdi_total` in the depression subset.

In [ ]:
outcome_vars = ["chose_gamble", "happiness", "choice_rt"]
predictor_vars = ["ev_advantage", "outcome", "rpe", "trial", "age_code", "female"]
df[outcome_vars + predictor_vars].describe().round(2)

## Grouping structure (for mixed-effects models in HW 3)

Trials are grouped within **plays**, and plays within **players**. Most players played once, but a third played more than once, so you have two levels of grouping.

In [ ]:
print("Plays per player:"); print(people["n_plays"].describe().round(1))
print("\nShare of players with more than one play:", round((people["n_plays"] > 1).mean(), 3))

## Time-to-event variables (for survival analysis in HW 3)

**Possible but approximate.** A natural question is how long players wait before playing again (*time* = days from the first play to the second; *event* = played again). The problem is censoring: the data don't say when each player's observation window ended, so players who never returned can't be given an accurate censoring time. Talk to the instructor before choosing this; a **mixed-effects model** is the more natural HW 3 analysis here.

In [ ]:
first_day = df[df["play"] == 1].groupby("player")["day_number"].first()
second_day = df[df["play"] == 2].groupby("player")["day_number"].first()
days_to_return = (second_day - first_day.reindex(second_day.index))
print("Players who returned:", len(days_to_return))
print(days_to_return.describe().round(1))

---
# Your Analysis Starts Here

> ✏️ **Your analysis.** Everything below this line is yours. Add as many cells as you need.

The dataset guide lists starter questions for each homework. 

## HW 1 — Describe

In [ ]:
# ✏️ Your HW 1 code here


## HW 2 — Predict / Moderate / Mediate

In [ ]:
# ✏️ Your HW 2 code here


## HW 3 — Mixed Effects / Survival

In [ ]:
# ✏️ Your HW 3 code here
